# Capstone pipeline

Full Capstone project requires:
1. Download DeepFashion images to train recognition of cloth type and condition.
2. Download Poshmark and Depop sample data for a few cloth types: price, brand, type, condition
3. Clean (remove missing) and unify (rename attributes to be the same across stores) data
4. Prepare data (extract features, prepare training, validation and testing datasets, as well as create synthetic data to similate conditions)
5. Train (run visual training, run categorical training)

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
import torch


# Finding where the notebook is located. The path is used for some more complicated script calls instead of making this notebook overgrown.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'README.md').exists() and (PROJECT_ROOT.parent / 'README.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

def load_env_file(env_path):
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

load_env_file(PROJECT_ROOT / '.env')

sys.path.append('scripts')
sys.path.append('src')
PYTHON = sys.executable

def run(args, check=True, env=None):
    if isinstance(args, str):
        print(f"$ {args}")
        return subprocess.run(args, shell=True, check=check, env=env)
    cmd = ' '.join(shlex.quote(str(part)) for part in args)
    print(f"$ {cmd}")
    return subprocess.run([str(part) for part in args], check=check, env=env)

print(PROJECT_ROOT)
print(PYTHON)


C:\Users\ionsphere\Downloads\ai course\capstone\pcmlai.capstone
C:\ProgramData\anaconda3\python.exe


In [2]:
DEMO_MODE = True
KAGGLE_DATASETS = [
    ('thusharanair/deepfashion2-original-with-dataframes', 'data/raw/deepfashion/kaggle_thushan'),
    ('vishalbsadanand/deepfashion-1', 'data/raw/deepfashion/kaggle_vishal'),
]
DEMO_KAGGLE_DATASETS = [
    ('vishalbsadanand/deepfashion-1', 'data/raw/deepfashion/kaggle_vishal'),
]

SCRAPE_JOBS = [
    {'query': 'dress', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'pants', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'shorts', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'jeans', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'skirt', 'max_items': 500, 'rate_limit': 2.0},
]

SYNTHETIC_INPUT_DIR = 'data/deepfashion/original'
SYNTHETIC_OUTPUT_DIR = 'data/deepfashion/synthetic_degraded'
SYNTHETIC_VARIATIONS = 5
SYNTHETIC_MAX_IMAGES = 5000

CLOTHING_TYPE_DIR = 'data/processed/clothing_type'
CONDITION_DIR = 'data/processed/condition_assessment'
MULTITASK_DIR = 'data/processed/multitask'
FEATURES_DIR = 'data/features'
EMBEDDINGS_DIR = 'data/embeddings'
PRICE_DATASET_DIR = 'data/price_classification'
VECTOR_INDEX_DIR = 'data/vector_index'

UNIFIED_DATA_CSV = 'data/scraped/unified_cleaned.csv'

CLOTHING_TYPE_MODEL_DIR = 'models/clothing_type'
CONDITION_MODEL_DIR = 'models/condition_assessment'
MULTITASK_MODEL_DIR = 'models/multitask'
PRICE_MODEL_DIR = 'models/price_model'

VISION_MODEL_FOR_FEATURES = f'{MULTITASK_MODEL_DIR}/checkpoints/best_loss.pth'

TRAIN_DEVICE = 'cuda'

In [7]:
# for scraper
run([PYTHON, '-m', 'playwright', 'install', 'chromium'])

$ 'C:\ProgramData\anaconda3\python.exe' -m playwright install chromium
$ pip install faiss-cpu


CompletedProcess(args=['pip', 'install', 'faiss-cpu'], returncode=0)

## Warning! Next operation will download ~56GBs of images

## Download

In [14]:
import download_deepfashion
import download_kaggle_datasets
import process_deepfashion_to_csv
import generate_statistics

# This takes several hours to receive all data in non-demo mode
#download_deepfashion.main(['--subset', 'all', '--download', '--extract', '--verify', *(['--demo'] if DEMO_MODE else [])]) # this can only be used with signed in google account. commenting it out for public repo.

# Don't forget to set KAGGLE_USERNAME and KAGGLE_KEY in system environment variables
active_kaggle_datasets = DEMO_KAGGLE_DATASETS if DEMO_MODE else KAGGLE_DATASETS
for dataset, output_dir in active_kaggle_datasets:
    args = ['--dataset', dataset, '--output', output_dir]
    if DEMO_MODE:
        args.append('--demo')
    download_kaggle_datasets.main(args)

process_deepfashion_to_csv.main(['--input', 'data/raw/deepfashion', '--output', 'data/processed'])
generate_statistics.main(['--processed-dir', 'data/processed', '--output', 'data/processed/deepfashion_statistics.csv'])

2026-03-26 21:59:58,647 - process_deepfashion_to_csv - INFO - Starting DeepFashion dataset processing...
2026-03-26 21:59:58,647 - process_deepfashion_to_csv - INFO - Input directory: data\raw\deepfashion
2026-03-26 21:59:58,647 - process_deepfashion_to_csv - INFO - Output directory: data\processed
2026-03-26 21:59:58,648 - process_deepfashion_to_csv - INFO - Processing Kaggle Thushan (DeepFashion2) dataset...
2026-03-26 21:59:58,648 - process_deepfashion_to_csv - INFO - Looking for CSV dataframes...
2026-03-26 22:00:01,048 - process_deepfashion_to_csv - INFO - Processing training data from: train.csv
2026-03-26 22:00:03,083 - process_deepfashion_to_csv - INFO - Loaded 312186 training samples
2026-03-26 22:00:03,084 - process_deepfashion_to_csv - INFO - Columns: ['path', 'segmentation', 'landmarks', 'b_box', 'category_id', 'category_name', 'scale', 'viewpoint', 'occlusion', 'zoom_in', 'img_height', 'img_width', 'split', 'source_dataset', 'subset']
2026-03-26 22:00:03,084 - process_deep

0

## Scrape

In [16]:
import scrape_marketplaces

# This takes several hours to receive all data

for job in SCRAPE_JOBS:
    scrape_marketplaces.main(['--query', job['query'], '--all-platforms', '--max-items', str(job['max_items']), '--rate-limit', str(job['rate_limit']), '--output-dir', 'data/raw/scraped'] + (['--demo'] if DEMO_MODE else []))

2026-03-26 22:00:57,087 - PoshmarkScraper - INFO - Starting Poshmark search scrape: 'dress' (max: 10)
2026-03-26 22:00:57,087 - PoshmarkScraper - INFO - Scraping page 1...


Demo mode enabled: max_items=10, rate_limit=0.5
Starting scrape: POSHMARK
Query: dress
Max items: 10
Rate limit: 0.5s between requests


2026-03-26 22:00:58,329 - PoshmarkScraper - INFO - Playwright browser initialized
2026-03-26 22:01:05,434 - PoshmarkScraper - WARNING - Timeout waiting for selector: [class*="tile"], [class*="card"], [class*="item"]
2026-03-26 22:01:05,666 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Dress-69bb3f40e2ac0ca27d85eba3
2026-03-26 22:01:11,207 - PoshmarkScraper - INFO - Successfully parsed item 69bb3f40e2ac0ca27d85eba3: Dress
2026-03-26 22:01:11,209 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Dress-69bb3f40e2ac0ca27d85eba3
2026-03-26 22:01:16,739 - PoshmarkScraper - INFO - Successfully parsed item 69bb3f40e2ac0ca27d85eba3: Dress
2026-03-26 22:01:16,740 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Dress-69bb3f40e2ac0ca27d85eba3
2026-03-26 22:01:22,269 - PoshmarkScraper - INFO - Successfully parsed item 69bb3f40e2ac0ca27d85eba3: Dress
2026-03-26 22:01:22,270 - PoshmarkScraper - INFO - Scraping item: https://poshmark


 Failed to scrape poshmark: 'PoshmarkScraper' object has no attribute '_browser_thread'

Starting scrape: THREDUP
Query: dress
Max items: 10
Rate limit: 0.5s between requests


2026-03-26 22:02:02,108 - ThredUpScraper - INFO - Playwright browser initialized
2026-03-26 22:02:02,174 - ThredUpScraper - INFO - GET https://www.thredup.com/search/dress?text=dress&page=1
2026-03-26 22:02:02,912 - ThredUpScraper - INFO - Status=200 Title=Used Dress On Sale Up To 90% Off | ThredUp
2026-03-26 22:02:07,429 - ThredUpScraper - WARNING - Saved Cloudflare debug artifacts to data\raw\scraped\debug_thredup
2026-03-26 22:02:07,646 - ThredUpScraper - INFO - Scraping page 2...
2026-03-26 22:02:07,698 - ThredUpScraper - INFO - GET https://www.thredup.com/search/dress?text=dress&page=2
2026-03-26 22:02:07,786 - ThredUpScraper - INFO - Status=403 Title=Just a moment...
2026-03-26 22:02:10,871 - ThredUpScraper - WARNING - Saved Cloudflare debug artifacts to data\raw\scraped\debug_thredup
2026-03-26 22:02:10,877 - ThredUpScraper - INFO - No more items found
2026-03-26 22:02:10,878 - ThredUpScraper - INFO - Completed scraping 0 items from ThredUp
2026-03-26 22:02:10,879 - DepopScraper

...


 Failed to scrape poshmark: 'PoshmarkScraper' object has no attribute '_browser_thread'

Starting scrape: THREDUP
Query: skirt
Max items: 10
Rate limit: 0.5s between requests


2026-03-26 22:13:54,824 - ThredUpScraper - INFO - Playwright browser initialized
2026-03-26 22:13:54,894 - ThredUpScraper - INFO - GET https://www.thredup.com/search/skirt?text=skirt&page=1
2026-03-26 22:13:55,591 - ThredUpScraper - INFO - Status=200 Title=Used Skirt On Sale Up To 90% Off | ThredUp
2026-03-26 22:14:00,108 - ThredUpScraper - WARNING - Saved Cloudflare debug artifacts to data\raw\scraped\debug_thredup
2026-03-26 22:14:00,387 - ThredUpScraper - INFO - Scraping page 2...
2026-03-26 22:14:00,421 - ThredUpScraper - INFO - GET https://www.thredup.com/search/skirt?text=skirt&page=2
2026-03-26 22:14:00,515 - ThredUpScraper - INFO - Status=403 Title=Just a moment...
2026-03-26 22:14:03,620 - ThredUpScraper - WARNING - Saved Cloudflare debug artifacts to data\raw\scraped\debug_thredup
2026-03-26 22:14:03,625 - ThredUpScraper - INFO - No more items found
2026-03-26 22:14:03,626 - ThredUpScraper - INFO - Completed scraping 0 items from ThredUp
2026-03-26 22:14:03,626 - DepopScraper


 Failed to scrape thredup: 'ThredUpScraper' object has no attribute '_browser_thread'

Starting scrape: DEPOP
Query: skirt
Max items: 10
Rate limit: 0.5s between requests


2026-03-26 22:14:04,172 - DepopScraper - INFO - Playwright browser initialized with anti-bot detection avoidance
2026-03-26 22:14:04,372 - DepopScraper - WARNING - API scraping failed, falling back to web scraping
2026-03-26 22:14:05,564 - DepopScraper - INFO - Waiting for products to load (10 seconds)...
2026-03-26 22:14:17,603 - DepopScraper - INFO - Found 24 product links
2026-03-26 22:14:17,608 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/sugarxlumshop33-cute-y2k-american-eagle-pink-e172/
2026-03-26 22:14:27,004 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/lexiswwielski-light-blue-denim-skirt-f03b/
2026-03-26 22:14:36,398 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/axnieorteja-leopard-print-floral-midi-skirt-7bd8/
2026-03-26 22:14:45,688 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/lfs305-navy-blue-lululemon-tennis-skirt-f66d/
2026-03-26 22:14:54,966 - DepopScraper - INFO - Scraping


 Failed to scrape depop: 'DepopScraper' object has no attribute '_browser_thread'

SCRAPING COMPLETE
Total items scraped: 0
  poshmark: 0 items
  thredup: 0 items
  depop: 0 items

Data saved to: data/raw/scraped


## Clean And Organize

In [10]:
import clean_scraped_data
import organize_data

clean_scraped_data.main(['--input-dir', 'data/raw/scraped', '--output-dir', 'data/scraped', '--remove-no-images'])
organize_data.main(['--create-dirs'])

# this step takes up to 2 hours due to GBs being copied
organize_data.main(['--organize-deepfashion', '--copy'])
organize_data.main(['--verify'])
organize_data.main([ '--unify', '--format', 'both'])
organize_data.main([ '--summary', '--output', 'data/data_organization_report.json'])


Marketplace Data Cleaning & Normalization
Loading data from: data\raw\scraped
Loading depop...


  depop batches: 100%|██████████| 5/5 [00:00<00:00, 90.36it/s]


Loading poshmark...


  poshmark batches: 100%|██████████| 5/5 [00:00<00:00, 90.65it/s]


Loaded 1547 total items
Initial dataset: 1547 items
Normalizing platform-specific data to unified schema...


  Normalizing records: 100%|██████████| 1547/1547 [00:00<00:00, 15591.91it/s]
2026-03-27 23:03:25,120 - INFO - Creating directory structure...


Normalized 1547 records to unified schema
Removing duplicates...
Removed 113 duplicates (7.3%)
Remaining items: 1434
Cleaning prices...
Removed 0 items with invalid prices
Price range: $3.00 - $950.00
Median price: $23.00
Cleaning text fields...
title: 1434/1434 (100.0%)
brand: 1309/1434 (91.3%)
category: 1432/1434 (99.9%)
condition: 1434/1434 (100.0%)
Standardizing conditions...
Standardized conditions:
Good: 947
New With Tags: 240
Like New: 210
New Without Tags: 32
Fair Condition: 3
Fair: 2
Validating image URLs...
Items with valid images: 1434/1434 (100.0%)
Filtering items without images...
Removed 0 items without images
Adding derived fields...
Saving cleaned unified data to: data\scraped
Saved unified CSV: data\scraped\unified_cleaned.csv
Saved full JSON: data\scraped\unified_cleaned.json
Saved depop: data\scraped\depop_cleaned.json (174 items)
Saved poshmark: data\scraped\poshmark_cleaned.json (1260 items)
Saved statistics: data\scraped\unified_stats.json
Unified Data Summary:
To

2026-03-27 23:03:25,121 - INFO -  Created: data\deepfashion\original
2026-03-27 23:03:25,122 - INFO -  Created: data\deepfashion\synthetic_degraded
2026-03-27 23:03:25,122 - INFO -  Created: data\deepfashion\metadata
2026-03-27 23:03:25,123 - INFO -  Created: data\scraped\poshmark
2026-03-27 23:03:25,124 - INFO -  Created: data\scraped\thredup
2026-03-27 23:03:25,125 - INFO -  Created: data\scraped\depop
2026-03-27 23:03:25,125 - INFO -  Created: data\scraped\unified
2026-03-27 23:03:25,126 - INFO -  Created: data\processed\train
2026-03-27 23:03:25,126 - INFO -  Created: data\processed\val
2026-03-27 23:03:25,126 - INFO -  Created: data\processed\test
2026-03-27 23:03:25,127 - INFO - Directory structure created successfully!
2026-03-27 23:03:25,127 - INFO - Organizing DeepFashion data...
2026-03-27 23:03:25,129 - INFO - Found 1 source directories:
2026-03-27 23:03:25,129 - INFO -   - kaggle_vishal
2026-03-27 23:03:25,129 - INFO - Target already exists: data\deepfashion\original\kaggle

## Prepare

In [21]:
import generate_synthetic_data
import prepare_splits
import prepare_clothing_type_dataset
import prepare_condition_dataset
import prepare_multitask_dataset

generate_synthetic_data.main(['--input-dir', SYNTHETIC_INPUT_DIR, '--output-dir', SYNTHETIC_OUTPUT_DIR, '--num-variations', str(SYNTHETIC_VARIATIONS), '--save-metadata', '--max-images', str(SYNTHETIC_MAX_IMAGES)])

# This steps creates separate Training, Validation and Testing sets. DeepFashion dataset already has its own separation. This split shuffles differently to mix in synthetic data.
# I've also discovered stratification as a method of fairer split regarding target category.
prepare_splits.main(['--all', '--clean', '--stratify', 'condition', '--report', 'data/processed/split_report.json'])
prepare_clothing_type_dataset.main(['--deepfashion-root', 'data/deepfashion', '--output-dir', CLOTHING_TYPE_DIR])
prepare_condition_dataset.main(['--deepfashion-dir', 'data/deepfashion/original', '--synthetic-dir', SYNTHETIC_OUTPUT_DIR, '--output-dir', CONDITION_DIR])
prepare_multitask_dataset.main(['--deepfashion2-csv', 'data/processed/deepfashion2_processed.csv', '--synthetic-root', SYNTHETIC_OUTPUT_DIR, '--output-dir', MULTITASK_DIR])


2026-03-27 08:33:57,383 - prepare_splits - INFO - Preparing directories for splits...
2026-03-27 08:33:57,383 - prepare_splits - WARNING - Removing existing directory: data\processed\train
2026-03-27 08:35:08,952 - prepare_splits - WARNING - Removing existing directory: data\processed\val
2026-03-27 08:35:08,953 - prepare_splits - WARNING - Removing existing directory: data\processed\test
2026-03-27 08:35:08,955 - prepare_splits - INFO - Directories prepared successfully!
2026-03-27 08:35:08,955 - prepare_splits - INFO - Splitting DeepFashion data...
2026-03-27 08:35:12,752 - prepare_splits - INFO - Found 54338 images to split
2026-03-27 08:35:12,753 - prepare_splits - INFO - Performing stratified split by: condition
2026-03-27 08:35:12,760 - prepare_splits - INFO - Found 10 groups for stratification
2026-03-27 08:35:12,795 - prepare_splits - INFO - Copying 38033 files to train set...
2026-03-27 08:37:32,004 - prepare_splits - INFO - Copied 38033 images to train set
2026-03-27 08:37:32

CLOTHING TYPE DATASET PREPARATION
Loading DeepFashion classification benchmark from: data\deepfashion\Anno
Loaded 20000 benchmark annotations
Removed 1563 items with unmapped categories
Total samples: 18,437
Unique categories: 13
Using official DeepFashion train/val/test benchmark splits
Train: 12,921 samples
Val: 1,846 samples
Test: 3,670 samples
Saved CSV files to data\processed\clothing_type
Saved category mapping to data\processed\clothing_type\category_mapping.json
Saved statistics to data\processed\clothing_type\dataset_statistics.json
DATASET SUMMARY
TRAIN:
Total samples: 12,921
Categories: 13
Top 5 categories:
 - dress: 4,707 (36.4%)
 - t-shirt: 2,347 (18.2%)
 - sweater: 1,192 (9.2%)
 - shirt: 1,055 (8.2%)
 - shorts: 880 (6.8%)
VAL:
Total samples: 1,846
Categories: 12
Top 5 categories:
 - dress: 690 (37.4%)
 - t-shirt: 318 (17.2%)
 - sweater: 160 (8.7%)
 - shirt: 156 (8.5%)
 - shorts: 129 (7.0%)
TEST:
Total samples: 3,670
Categories: 13
Top 5 categories:
 - dress: 1,282 (34.9%)

Original: 100%|██████████| 10000/10000 [00:00<00:00, 1532053.91it/s]


Processing synthetic degraded images...
Collected 10000 samples
Creating stratified splits...
Train: 7000 samples (70.0%)
Val: 1500 samples (15.0%)
Test: 1500 samples (15.0%)
Saved: data\processed\condition_assessment\train.csv
Saved: data\processed\condition_assessment\val.csv
Saved: data\processed\condition_assessment\test.csv
Saved: data\processed\condition_assessment\dataset_stats.json
DATASET PREPARATION COMPLETE
Dataset saved to: data/processed/condition_assessment
Saved multitask dataset to C:\Users\ionsphere\Downloads\ai course\capstone\pcmlai.capstone\data\processed\multitask
Rows: 191961
Categories: 8


## Train Vision Models

In [ ]:
import train_clothing_type
import train_condition_assessment
import train_multitask_model

# first attempt to have individual models per task: cloth type
train_clothing_type.main(['--data-dir', CLOTHING_TYPE_DIR, '--output-dir', CLOTHING_TYPE_MODEL_DIR, '--device', TRAIN_DEVICE, *(['--demo'] if DEMO_MODE else [])])
# and condition
train_condition_assessment.main(['--train-csv', f'{CONDITION_DIR}/train.csv', '--val-csv', f'{CONDITION_DIR}/val.csv', '--data-dir', str(PROJECT_ROOT), '--output-dir', CONDITION_MODEL_DIR, *(['--demo'] if DEMO_MODE else [])])

history_path = r"models\condition_assessment\training_history.json"
with open(history_path, "r") as f:
    history = json.load(f)

df = pd.json_normalize(history)

plot_df = pd.concat(
    [
        df[["epoch", "train.loss", "train.mae", "train.rmse"]]
        .rename(columns=lambda c: c.replace("train.", "") if c != "epoch" else c)
        .assign(split="Train"),
        df[["epoch", "val.loss", "val.mae", "val.rmse"]]
        .rename(columns=lambda c: c.replace("val.", "") if c != "epoch" else c)
        .assign(split="Validation"),
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
metrics = ["loss", "mae", "rmse"]
titles = ["Loss", "MAE", "RMSE"]

for ax, metric, title in zip(axes.flat, metrics, titles):
    sns.lineplot(
        data=plot_df,
        x="epoch",
        y=metric,
        hue="split",
        marker="o",
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)

plt.show()


# second attempt: train one vision model to categorize both cloth type and condition
train_multitask_model.main(['--train-csv', f'{MULTITASK_DIR}/train.csv', '--val-csv', f'{MULTITASK_DIR}/val.csv', '--output-dir', MULTITASK_MODEL_DIR, *(['--demo'] if DEMO_MODE else [])])

history_path = r"models\multitask\training_history.json"

with open(history_path, "r") as f:
    history = json.load(f)

df = pd.DataFrame({
    "epoch": range(1, len(history["train_loss"]) + 1),
    "train_loss": history["train_loss"],
    "val_loss": history["val_loss"],
    "train_accuracy": history["train_accuracy"],
    "val_accuracy": history["val_accuracy"],
    "train_mae": history["train_condition_mae"],
    "val_mae": history["val_condition_mae"],
})

plot_df = pd.concat(
    [
        df[["epoch", "train_loss", "train_accuracy", "train_mae"]]
        .rename(columns={
            "train_loss": "loss",
            "train_accuracy": "accuracy",
            "train_mae": "mae",
        })
        .assign(split="Train"),
        df[["epoch", "val_loss", "val_accuracy", "val_mae"]]
        .rename(columns={
            "val_loss": "loss",
            "val_accuracy": "accuracy",
            "val_mae": "mae",
        })
        .assign(split="Validation"),
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
metrics = ["loss", "accuracy", "mae"]
titles = ["Loss", "Accuracy", "MAE"]

for ax, metric, title in zip(axes, metrics, titles):
    sns.lineplot(
        data=plot_df,
        x="epoch",
        y=metric,
        hue="split",
        marker="o",
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)

plt.show()

Demo mode enabled (2 epochs on the real dataset)
Loaded 13 categories
Image root: data\deepfashion
Loading datasets...
Train dataset: 12921 samples
Val dataset: 1846 samples
CUDA not available, using CPU (this will be slow)


C:\Users\ionsphere\Downloads\ai course\capstone\pcmlai.capstone\scripts\train_clothing_type.py:135: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None
C:\ProgramData\anaconda3\Lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


Model Parameters:
Total: 18,675,798
Trainable: 18,675,798
Optimizer: AdamW (lr=0.0001, wd=1e-05)
Scheduler: StepLR
STARTING TRAINING
Epochs: 2
Batch size: 32
Train batches: 404
Val batches: 58


C:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  torch.utils.data.graph_settings.apply_sharding(
C:\Users\ionsphere\Downloads\ai course\capstone\pcmlai.capstone\scripts\train_clothing_type.py:225: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\ProgramData\anaconda3\Lib\site-packages\torch\cuda\amp\autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  


## Build features for price dataset

In [9]:
from src.models.price import FeatureExtractor

data_csv = Path(UNIFIED_DATA_CSV)
if not data_csv.exists():
    raise FileNotFoundError(data_csv)

vision_model_path = Path(VISION_MODEL_FOR_FEATURES)
if not vision_model_path.exists():
    raise FileNotFoundError(vision_model_path)

vision_embeddings = None

batch_size = 32
device = 'cpu'
output_dir = Path(FEATURES_DIR)

output_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_csv)

required_cols = {"title", "brand", "category", "condition", "price"}
missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

feature_extractor = FeatureExtractor(
    text_features=False,
    categorical_features=True,
    device=device,
)

data_dict = {
    "title": df["title"].fillna("").astype(str).tolist(),
    "brand": df["brand"].fillna("unknown").astype(str).tolist(),
    "category": df["category"].fillna("unknown").astype(str).tolist(),
    "condition": df["condition"].fillna("good").astype(str).tolist(),
    "condition_score": df.get("condition_score", pd.Series([5.0] * len(df))).fillna(5.0).values,
}
features, feature_names = feature_extractor.extract_all_features(data_dict, images=None, fit=True)

np.save(output_dir / "features.npy", features)
with open(output_dir / "feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)
feature_extractor.save(output_dir / "feature_extractor.pkl")

metadata = {
    "n_samples": int(features.shape[0]),
    "n_features": int(features.shape[1]),
    "has_vision_embeddings": vision_embeddings is not None,
    "vision_embedding_dim": int(vision_embeddings.shape[1]) if vision_embeddings is not None else 0,
    "feature_names": feature_names,
    "data_csv": str(data_csv),
    "vision_model": str(vision_model_path) if vision_model_path else None,
}
with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

## Train Price Model

In [15]:
from src.models.price import PriceRangeBinner
from src.models.price import PriceClassifier

DEFAULT_BINNING_STRATEGY = 'quantile'
DEFAULT_BIN_COUNT = 5
DEFAULT_TRAIN_RATIO = 0.7
DEFAULT_VAL_RATIO = 0.15
DEFAULT_TEST_RATIO = 0.15
DEFAULT_RANDOM_SEED = 42

def make_json_serializable(value):
    if isinstance(value, dict):
        return {str(k): make_json_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_serializable(v) for v in value]
    if isinstance(value, np.ndarray):
        return [make_json_serializable(v) for v in value.tolist()]
    if isinstance(value, np.generic):
        return value.item()
    return value

data_csv = Path(UNIFIED_DATA_CSV)
output_dir = Path(PRICE_DATASET_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
    
if not data_csv.exists():
    raise FileNotFoundError(f"Data CSV not found: {data_csv}")
    
df = pd.read_csv(data_csv)
if 'price' not in df.columns:
    raise ValueError("'price' column not found in data CSV")
    
prices = df['price'].values

stats = {
    'count': int(len(prices)),
    'mean': float(np.mean(prices)),
    'median': float(np.median(prices)),
    'std': float(np.std(prices)),
    'min': float(np.min(prices)),
    'max': float(np.max(prices)),
    'q25': float(np.percentile(prices, 25)),
    'q75': float(np.percentile(prices, 75))
}

print(f"Price Statistics:")
print(f"Count: ${stats['count']}")
print(f"Mean: ${stats['mean']:.2f}")
print(f"Median: ${stats['median']:.2f}")
print(f"Std Dev: ${stats['std']:.2f}")
print(f"Min: ${stats['min']:.2f}")
print(f"Max: ${stats['max']:.2f}")
print(f"Q25: ${stats['q25']:.2f}")
print(f"Q75: ${stats['q75']:.2f}")

category_col = None
for col in ["category_name", "category", "category_id"]:
    if col in df.columns:
        category_col = col
        break

if category_col in df.columns:
    category_stats = df.groupby(category_col)['price'].agg([
        'count', 'mean', 'median', 'std', 'min', 'max'
    ]).round(2)
    print(f"Price by Category:")
    print(category_stats)
    
    category_stats.to_csv(output_dir / 'price_by_category.csv')

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].hist(prices, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Price Distribution')
axes[0, 0].axvline(stats['mean'], color='red', linestyle='--', label=f"Mean: ${stats['mean']:.2f}")
axes[0, 0].axvline(stats['median'], color='green', linestyle='--', label=f"Median: ${stats['median']:.2f}")
axes[0, 0].legend()

axes[0, 1].hist(prices, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Price ($)')
axes[0, 1].set_ylabel('Frequency (log scale)')
axes[0, 1].set_title('Price Distribution (Log Scale)')
axes[0, 1].set_yscale('log')

axes[1, 0].boxplot(prices, vert=True)
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].set_title('Price Box Plot')
axes[1, 0].grid(axis='y', alpha=0.3)

if category_col in df.columns:
    categories = df[category_col].unique()[:10]  # Limit to 10 categories for readability
    category_prices = [df[df[category_col] == cat]['price'].values for cat in categories]
    axes[1, 1].boxplot(category_prices, labels=categories, vert=True)
    axes[1, 1].set_xlabel('Category')
    axes[1, 1].set_ylabel('Price ($)')
    axes[1, 1].set_title('Price by Category (Top 10)')
    axes[1, 1].tick_params(axis='x', rotation=45)
else:
    axes[1, 1].text(0.5, 0.5, 'Category data not available', 
                   ha='center', va='center', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.savefig(output_dir / 'price_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"Saved price distribution plot to {output_dir / 'price_distribution.png'}")

# Save statistics
with open(output_dir / 'price_statistics.json', 'w') as f:
    json.dump(stats, f, indent=2)

prices = df['price'].values
categories = None

binner = PriceRangeBinner(
    strategy=DEFAULT_BINNING_STRATEGY,
    n_bins=DEFAULT_BIN_COUNT,
    custom_bins=None,
    category_specific=False
)

bin_indices, bin_labels = binner.fit_transform(prices, categories)

df['price_bin'] = bin_indices
df['price_bin_label'] = bin_labels

print(f"Price Bins:")
bin_info = binner.get_bin_info()
for cat, bins in bin_info['bins'].items():
    bin_label_names = bin_info['labels'][cat]
    print(f"Category: {cat}")
    for i, (label, bin_start, bin_end) in enumerate(zip(bin_label_names, bins[:-1], bins[1:])):
        count = np.sum((bin_indices == i) & ((categories == cat) if categories is not None else True))
        print(f"Bin {i}: {label} (n={count})")

fig, ax = plt.subplots(figsize=(12, 6))

bin_counts = pd.Series(bin_labels).value_counts().sort_index()
ax.bar(range(len(bin_counts)), bin_counts.values, edgecolor='black', alpha=0.7)
ax.set_xlabel('Price Bin')
ax.set_ylabel('Number of Samples')
ax.set_title('Distribution of Samples Across Price Bins')
ax.set_xticks(range(len(bin_counts)))
ax.set_xticklabels(bin_counts.index, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

for i, v in enumerate(bin_counts.values):
    ax.text(i, v + max(bin_counts.values) * 0.01, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.savefig(output_dir / 'bin_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

serializable_bin_info = make_json_serializable(bin_info)
with open(output_dir / 'price_binner.json', 'w') as f:
    json.dump(serializable_bin_info, f, indent=2)

train_ratio = DEFAULT_TRAIN_RATIO
val_ratio = DEFAULT_VAL_RATIO
test_ratio = DEFAULT_TEST_RATIO
random_seed = DEFAULT_RANDOM_SEED

target_labels = bin_indices

X_trainval, X_test, y_trainval, y_test, idx_trainval, idx_test = train_test_split(
    features, target_labels, np.arange(len(target_labels)),
    test_size=test_ratio,
    stratify=target_labels,
    random_state=random_seed
)

val_size = val_ratio / (train_ratio + val_ratio)
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_trainval, y_trainval, idx_trainval,
    test_size=val_size,
    stratify=y_trainval,
    random_state=random_seed
)

splits = {
    'train': (X_train, y_train, idx_train),
    'val': (X_val, y_val, idx_val),
    'test': (X_test, y_test, idx_test)
}

for split_name, (X, y, idx) in splits.items():
    np.save(output_dir / f'{split_name}_features.npy', X)
    np.save(output_dir / f'{split_name}_labels.npy', y)
    np.save(output_dir / f'{split_name}_indices.npy', idx)
    
    split_df = df.iloc[idx].copy()
    split_df.to_csv(output_dir / f'{split_name}_data.csv', index=False)
    
    print(f"{split_name.capitalize()} split: {len(X)} samples")
    
    unique, counts = np.unique(y, return_counts=True)
    print(f"Label distribution: {dict(zip(unique, counts))}")

metadata = {
    'train_ratio': train_ratio,
    'val_ratio': val_ratio,
    'test_ratio': test_ratio,
    'random_seed': random_seed,
    'n_train': int(len(X_train)),
    'n_val': int(len(X_val)),
    'n_test': int(len(X_test)),
    'n_features': int(features.shape[1]),
    'n_classes': int(len(np.unique(target_labels)))
}

with open(output_dir / 'split_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

price_dir = Path(PRICE_DATASET_DIR)

X_train = np.load(price_dir / 'train_features.npy')
y_train = np.load(price_dir / 'train_labels.npy')
X_val = np.load(price_dir / 'val_features.npy')
y_val = np.load(price_dir / 'val_labels.npy')
X_test = np.load(price_dir / 'test_features.npy')
y_test = np.load(price_dir / 'test_labels.npy')

with open(price_dir / 'price_binner.json', 'r') as f:
    binner_info = json.load(f)

with open(price_dir / 'split_metadata.json', 'r') as f:
    split_info = json.load(f)

feature_names = None
feature_names_path = price_dir.parent / 'features' / 'feature_names.json'
if feature_names_path.exists():
    with open(feature_names_path, 'r') as f:
        feature_names = json.load(f)

bins_dict = binner_info.get('bins', {})
labels_dict = binner_info.get('labels', {})

if 'global' in labels_dict:
    class_names = labels_dict['global']
else:
    first_cat = list(labels_dict.keys())[0]
    class_names = labels_dict[first_cat]

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Val: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"Classes: {len(class_names)} price ranges")
print(f"Class names: {class_names}")

data = {
    'X_train': X_train,
    'y_train': y_train,
    'X_val': X_val,
    'y_val': y_val,
    'X_test': X_test,
    'y_test': y_test,
    'class_names': class_names,
    'feature_names': feature_names,
    'n_classes': len(class_names),
    'binner_info': binner_info,
    'split_info': split_info
}
    
output_dir = Path(PRICE_MODEL_DIR)

models_config = {
    'xgboost': {
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'random_state': 42
    },
    'lightgbm': {
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'random_state': 42,
        'verbose': -1
    },
    'random_forest': {
        'n_estimators': 100,
        'max_depth': 10,
        'random_state': 42,
        'n_jobs': -1
    }
}

results = []

for model_type, model_params in models_config.items():
    try:
        model = PriceClassifier(
            model_type=model_type,
            n_classes=data['n_classes'],
            **model_params
        )
        
        # Train
        model.train(
            data['X_train'], data['y_train'],
            data['X_val'], data['y_val'],
        )
        val_metrics = model.evaluate(
            data['X_val'],
            data['y_val'],
            class_names=data['class_names']
        )
        
        print(f"Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"F1-score (weighted): {val_metrics['f1_weighted']:.4f}")
        print(f"Within +/-1 bracket: {val_metrics['within_1_accuracy']:.4f}")
        
        test_metrics = model.evaluate(
            data['X_test'],
            data['y_test'],
            class_names=data['class_names']
        )
        
        print(f"Accuracy: {test_metrics['accuracy']:.4f}")
        print(f"F1-score (weighted): {test_metrics['f1_weighted']:.4f}")
        print(f"Within +/-1 bracket: {test_metrics['within_1_accuracy']:.4f}")
        
        results.append({
            'model': model,
            'model_type': model_type,
            'val_metrics': val_metrics,
            'test_metrics': test_metrics
        })
    except Exception as e:
        print(f"Error training {model_type}: {e}")
        continue

print("MODEL COMPARISON RESULTS")

comparison_data = []
for result in results:
    comparison_data.append({
        'Model': result['model_type'].upper(),
        'Val Accuracy': f"{result['val_metrics']['accuracy']:.4f}",
        'Val F1': f"{result['val_metrics']['f1_weighted']:.4f}",
        'Val +/-1': f"{result['val_metrics']['within_1_accuracy']:.4f}",
        'Test Accuracy': f"{result['test_metrics']['accuracy']:.4f}",
        'Test F1': f"{result['test_metrics']['f1_weighted']:.4f}",
        'Test +/-1': f"{result['test_metrics']['within_1_accuracy']:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

comparison_df.to_csv(output_dir / 'model_comparison.csv', index=False)
print(f"Saved comparison to {output_dir / 'model_comparison.csv'}")

best_result = max(results, key=lambda x: x['test_metrics']['accuracy'])
print(f"Best Model: {best_result['model_type'].upper()}")
print(f"Test Accuracy: {best_result['test_metrics']['accuracy']:.4f}")
print(f"Test F1: {best_result['test_metrics']['f1_weighted']:.4f}")
print(f"Within +/-1 bracket: {best_result['test_metrics']['within_1_accuracy']:.4f}")


Price Statistics:
Count: $1434
Mean: $32.13
Median: $23.00
Std Dev: $38.90
Min: $3.00
Max: $950.00
Q25: $15.00
Q75: $35.00
Price by Category:
                           count   mean  median     std    min     max
category                                                              
Bags                           1  12.24   12.24     NaN  12.24   12.24
Dresses                      273  39.92   32.00   28.26   3.85  199.00
Jeans                        217  31.73   25.00   23.07   5.00  142.00
Pants                         29  16.82   16.45    5.81   4.11   26.95
Pants & Jumpsuits            171  34.49   28.00   24.30   3.00  200.00
Shorts                       192  18.65   15.00   15.52   3.32  150.00
Skirts                       209  27.25   19.00   36.11   5.95  310.00
Tops                           1  13.30   13.30     NaN  13.30   13.30
Women > Dresses              108  41.52   32.00   30.64   5.00  130.00
Women > Jeans                 48  36.08   25.00   33.35  10.00  175.00
Women 

C:\Users\ionsphere\AppData\Local\Temp\ipykernel_35416\552369491.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[1, 1].boxplot(category_prices, labels=categories, vert=True)


Saved price distribution plot to data\price_classification\price_distribution.png
Price Bins:
Category: global
Bin 0: $3-$14 (n=277)
Bin 1: $14-$20 (n=291)
Bin 2: $20-$27 (n=277)
Bin 3: $27-$40 (n=293)
Bin 4: $40-$950 (n=296)
Train split: 1003 samples
Label distribution: {np.int64(0): np.int64(194), np.int64(1): np.int64(203), np.int64(2): np.int64(194), np.int64(3): np.int64(205), np.int64(4): np.int64(207)}
Val split: 215 samples
Label distribution: {np.int64(0): np.int64(41), np.int64(1): np.int64(44), np.int64(2): np.int64(41), np.int64(3): np.int64(44), np.int64(4): np.int64(45)}
Test split: 216 samples
Label distribution: {np.int64(0): np.int64(42), np.int64(1): np.int64(44), np.int64(2): np.int64(42), np.int64(3): np.int64(44), np.int64(4): np.int64(44)}
Train: 1003 samples, 4 features
Val: 215 samples
Test: 216 samples
Classes: 5 price ranges
Class names: ['$3-$14', '$14-$20', '$20-$27', '$27-$40', '$40-$950']
[0]	validation_0-mlogloss:1.59515
[1]	validation_0-mlogloss:1.58326


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.3302
F1-score (weighted): 0.3267
Within +/-1 bracket: 0.7116
Accuracy: 0.2546
F1-score (weighted): 0.2526
Within +/-1 bracket: 0.6481
MODEL COMPARISON RESULTS
        Model Val Accuracy Val F1 Val +/-1 Test Accuracy Test F1 Test +/-1
      XGBOOST       0.3070 0.3033   0.6605        0.2731  0.2631    0.6620
     LIGHTGBM       0.2837 0.2816   0.6744        0.2731  0.2653    0.7037
RANDOM_FOREST       0.3302 0.3267   0.7116        0.2546  0.2526    0.6481
Saved comparison to models\price_model\model_comparison.csv
Best Model: XGBOOST
Test Accuracy: 0.2731
Test F1: 0.2631
Within +/-1 bracket: 0.6620


## Generate Embeddings

In [5]:
from src.models.embeddings import (
    VisionEmbeddingExtractor,
    TextEmbeddingExtractor,
    MultiModalEmbedding,
    EmbeddingPipeline,
)

df = pd.read_csv(UNIFIED_DATA_CSV)
images = None
texts = None

# Prepare images
if 'image' not in df.columns:
    print(f"Warning: Image column 'image' not found in CSV")
else:
    images = df['image'].tolist()
    print(f"Prepared {len(images)} images")

# Prepare texts
text_parts = []

# Add title if available
if 'title' in df.columns:
    titles = df['title'].fillna('').astype(str)
    text_parts.append(titles)

# Combine text parts
if text_parts:
    texts = [' '.join(parts) for parts in zip(*text_parts)]
    print(f"Prepared {len(texts)} texts")

print("INITIALIZING EMBEDDING PIPELINE")
pipeline = EmbeddingPipeline(
    vision_model_path=VISION_MODEL_FOR_FEATURES,
    text_model_name=None,
    fusion_method='concat',
    reduce_dim=True,
    reduction_method='pca',
    target_dim=128,
    device=TRAIN_DEVICE,
)

print(f"Embedding dimension: {pipeline.multi_modal.embedding_dim}")
print(f"Will be reduced to: 128")

print("GENERATING EMBEDDINGS")

embeddings = pipeline.generate(
    images=images,
    texts=texts,
    batch_size=32,
    normalize=True,
)

print(f"Generated embeddings shape: {embeddings.shape}")

metadata = {
    'mode': 'multimodal',
    'num_samples': len(embeddings),
    'embedding_dim': embeddings.shape[1],
    'vision_model': VISION_MODEL_FOR_FEATURES,
    'text_model': None,
    'fusion_method': 'concat',
    'reduced': True,
    'reduction_method': 'pca',
    'target_dim': 128,
    'normalized': True,
    'generated_at': datetime.now().isoformat(),
    'data_source': UNIFIED_DATA_CSV,
    'item_ids': df.get('item_id', df.index).tolist(),
}

print("SAVING EMBEDDINGS")

pipeline.save(EMBEDDINGS_DIR, embeddings, metadata)

df_path = Path(EMBEDDINGS_DIR) / 'items_data.csv'
df.to_csv(df_path, index=False)


Prepared 1434 images
Prepared 1434 texts
INITIALIZING EMBEDDING PIPELINE
Embedding dimension: 1792
Will be reduced to: 128
GENERATING EMBEDDINGS


Loading images:   0%|          | 0/1434 [00:00<?, ?it/s]


OSError: [Errno 22] Invalid argument: 'https://media-photos.depop.com/b1/277949866/3564220301_ee1ccb0cc4154ccc9a3ee10d8a3c69bf/P0.jpg'

## Build Vector Index

In [8]:
from src.vector_search import FAISSIndex, SimilaritySearch

embeddings_dir = Path(EMBEDDINGS_DIR)
embeddings_path = embeddings_dir / 'embeddings.npy'
if not embeddings_path.exists():
    raise FileNotFoundError(f"Embeddings not found at {embeddings_path}")

embeddings = np.load(str(embeddings_path))
print(f"Loaded embeddings: shape={embeddings.shape}")

metadata_path = embeddings_dir / 'metadata.json'
if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    item_ids = metadata.get('item_ids', [])
else:
    item_ids = list(range(len(embeddings)))

items_csv = embeddings_dir / 'items_data.csv'
if items_csv.exists():
    items_data = pd.read_csv(items_csv)
    print(f"Loaded items data: {len(items_data)} items")
else:
    items_data = pd.DataFrame({'item_id': item_ids})
    print("Warning: items_data.csv not found, using minimal item IDs")

index_metadata = [{'item_id': item_id} for item_id in item_ids]

print(f"Building {index_type.upper()} index...")
print(f"Metric: {metric}")
print(f"Embeddings: {embeddings.shape}")

start_time = time.time()
index_type = 'flat'
metric = 'cosine'
nlist = 100
nprobe = 10
hnsw_m = 32
use_gpu = True

index = FAISSIndex(
    index_type=index_type,
    dimension=embeddings.shape[1],
    metric=metric,
    nlist=nlist,
    nprobe=nprobe,
    hnsw_m=hnsw_m,
    use_gpu=use_gpu,
)
index.build(embeddings, metadata)

build_time = time.time() - start_time
print(f"Index built in {build_time:.2f} seconds")

stats = index.get_stats()
print("Index Statistics:")
for key, value in stats.items():
    print(f"{key}: {value}")

output_dir = Path(VECTOR_INDEX_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'clothing_index'
index.save(str(output_path))
items_csv_path = output_dir / 'items_data.csv'
items_data.to_csv(items_csv_path, index=False)
print(f"Saved items data to {items_csv_path}")
print(f"Index saved to {output_path}")


FileNotFoundError: Embeddings not found at data\embeddings\embeddings.npy